# Causal estimator benchmark: mboostrs against EconML and DoubleML

An earlier causal battery scored mboostrs below EconML's linear DML and below a
hand-built R-learner. The analysts in that battery used different nuisance learners,
second stages and defaults, so the gap could not be assigned to a cause. This study
fixes those choices, runs each library's estimator on the same draws, and measures
each arm's error. It asks three questions:

1. How far mboostrs' causal estimators are from EconML and DoubleML when the nuisance
   learners and confounder set are matched, with the fold and response-weight
   differences listed below.
2. Which of four sources accounts for the gap: the nuisance learners, the estimation
   framework, the second-stage form, or the defaults.
3. Whether mboostrs is faster than the other libraries at comparable settings.

The protocol is `docs/book/benchmarks/causal_estimators/PROTOCOL.md`. The per-draw
records are in `per_draw_results.jsonl`, and this page recomputes its tables from
them.

## Design

The data are the job-training design of the causal battery: a binary program
indicator and an earnings outcome, with confounders drawn from real survey records. The
truth is known for each draw, because the generator sets the average treatment effect
(ATE) and the individual effects. The study has twelve draws: ten with a true effect
and two null draws with no effect. Every arm sees the same data draw and seed. The mboostrs folds differ from the
other libraries' folds, as recorded in the protocol and limitations below.

The arms cover four estimators:

- **mboostrs** AIPW (augmented inverse probability weighting), its effect models and its
  rank and modifier calls. Its variants change the defaults one at a time.
- **EconML** `LinearDML` and `CausalForestDML`, each with gradient-boosted (HGB) or GAM
  nuisance learners.
- **DoubleML** `IRM`, with HGB or GAM nuisances.
- A **hand-built** R-learner, linear or with HGB or GAM nuisances.

The estimator arms report an ATE estimate and, where available, a 90% interval.
The separate `mb_rank` and `mb_modifiers` calls report rankings and modifier
assessments; they have no ATE fields.

In [ ]:
import json
from pathlib import Path

import polars as pl
from great_tables import GT
from IPython.display import display

# This page runs from docs/book/benchmarks; the per-draw records sit beside the protocol.
DRAWS = Path("causal_estimators/per_draw_results.jsonl")
RESULTS = Path("causal_estimators/results.json")

records = [json.loads(line) for line in DRAWS.read_text().splitlines()]
per_draw = pl.DataFrame(
    [
        {
            "draw": r["draw"],
            "arm": r["arm"],
            "null": bool(r["null"]),
            "ate_hat": r.get("ate_hat"),
            "ate_truth": r.get("ate_truth"),
            "ate_error": r.get("ate_error"),
            "ate_se": r.get("ate_se"),
            "wall_s": r["wall_s"],
            "error": r["error"],
        }
        for r in records
    ]
)
print(f"{per_draw.height} draw-arm records; arms: {per_draw['arm'].n_unique()}; records with an error: {per_draw['error'].is_not_null().sum()}")

# Ranking and modifier calls intentionally report no ATE. Keep their records visible.
non_ate_arms = ["mb_rank", "mb_modifiers"]
record_status = per_draw.group_by("arm").agg(
    pl.len().alias("Records"),
    pl.col("ate_hat").count().alias("ATE estimates"),
    pl.col("error").is_not_null().sum().alias("Failed calls"),
).sort("arm")
display(GT(record_status).tab_header(
    title="Recorded calls and available ATE estimates",
    subtitle="Twelve draws per arm, including two null draws; ranking and modifier calls have no ATE",
))

The first check is whether the arms ran. The cell above counts the records and the
errors. Every arm ran on every draw, and no record carries an error.

## Error on the effect draws, by arm

The table uses the ten effect draws. It gives the median and the largest absolute ATE
error, in dollars, across those draws, and the median fit time in seconds. Bold marks
the lowest median error in the table. Lower is better.

The `mb_aipw_ate` row is the AIPW default arm. It was fitted with `smooth=False`, which was the default of `aipw` when the arm was recorded. The current default is `smooth=True`, so `aipw` now fits the smooth arm, whose row is `mb_aipw_smooth`. The rerun table below also predates that change, so its `mb_aipw_ate` column was fitted with `smooth=False` as well.

In [ ]:
# Exclude calls whose task was ranking or modifier assessment, rather than ATE estimation.
effect = per_draw.filter(~pl.col("null") & ~pl.col("arm").is_in(non_ate_arms))
arm_error = (
    effect.with_columns(pl.col("ate_error").abs().alias("abs_error"))
    .group_by("arm")
    .agg(
        pl.col("abs_error").median().alias("Median absolute error"),
        pl.col("abs_error").max().alias("Largest absolute error"),
        pl.col("wall_s").median().alias("Median seconds"),
        pl.len().alias("Draws"),
        (pl.col("ate_hat").is_not_null() & pl.col("error").is_null()).sum().alias("Completed"),
    )
    # A failed or missing estimate makes the full-arm error summary unavailable.
    .with_columns([
        pl.when(pl.col("Completed") == pl.col("Draws")).then(pl.col(column)).otherwise(None).alias(column)
        for column in ["Median absolute error", "Largest absolute error", "Median seconds"]
    ])
    .sort("Median absolute error", nulls_last=True)
)

best_median = arm_error["Median absolute error"].min()
arm_rows = []
for r in arm_error.iter_rows(named=True):
    median_text = "unavailable" if r["Median absolute error"] is None else f"{r['Median absolute error']:,.0f}"
    if best_median is not None and r["Median absolute error"] == best_median:
        median_text = f"**{median_text}**"
    arm_rows.append(
        {
            "Arm": r["arm"],
            "Median absolute error ($)": median_text,
            "Largest absolute error ($)": "unavailable" if r["Largest absolute error"] is None else f"{r['Largest absolute error']:,.0f}",
            "Median seconds": "unavailable" if r["Median seconds"] is None else f"{r['Median seconds']:.2f}",
            "Draws": r["Draws"],
            "Completed": r["Completed"],
        }
    )

arm_table = pl.DataFrame(arm_rows)
(
    GT(arm_table)
    .tab_header(
        title="ATE error by arm on the ten effect draws (recorded run)",
        subtitle="Recorded run, October 11 build; absolute error of the ATE estimate in dollars; lower is better",
    )
    .fmt_markdown(columns=["Median absolute error ($)"])
    .tab_stubhead(label="Arm")
    .tab_source_note("Median and largest across all ten effect draws; an incomplete arm has unavailable summaries. Ranking/modifier calls are listed separately above. Seconds are median fit and predict time. Source: per_draw_results.jsonl.")
)

Six arms have a median error between 330 and 500 dollars: the EconML and hand-built
linear arms with HGB or GAM nuisances, and mboostrs' smooth AIPW. In the recorded run,
four mboostrs AIPW arms had a median error of about 294,000 dollars, about 900 times the
error of the best arm. The next sections show that this recorded result came from the
library build of October 11, and that the current build does not reproduce it.

## The default AIPW arm, draw by draw

The next table shows the ATE estimate on each effect draw for the default mboostrs
AIPW arm, the smooth variant, and EconML's linear arm with HGB nuisances. Each row is
one draw, and the truth is the same for every arm on that draw.

In [ ]:
focus = ["mb_aipw_ate", "mb_aipw_smooth", "ec_lin_hgb"]
wide = (
    effect.filter(pl.col("arm").is_in(focus))
    .select(["draw", "arm", "ate_hat", "ate_truth"])
    .pivot(on="arm", index=["draw", "ate_truth"], values="ate_hat")
    .sort("draw")
)
(
    GT(wide)
    .tab_header(
        title="ATE estimates on each effect draw",
        subtitle="Dollars of earnings per program participant; truth is the draw's true ATE",
    )
    .fmt_number(columns=["ate_truth"] + focus, decimals=0)
    .cols_label(ate_truth="True ATE", draw="Draw")
    .tab_stubhead(label="Draw")
    .tab_source_note("Source: per_draw_results.jsonl. Recorded mb_aipw_ate estimates range from about 16,500 to 19.1 million dollars over the ten effect draws.")
)

In the recorded run, the default AIPW estimates are not close to the truth on any effect
draw. They range from about 16,000 to about 19 million dollars. The reported standard
errors are very large, with a mean of about 2.2 million dollars, so the 90% intervals are
very wide. That is why the summary's coverage for this arm is 1.0. The interval contains
the truth only because it is wide, and coverage is not evidence that the estimate is
correct.

In the recorded run the same pattern appears in `mb_aipw_noclip`, `mb_aipw_notune` and
`mb_aipw_proptrees`, with median errors of about 284,000 to 294,000 dollars. The arms
`mb_aipw_smooth` and `mb_aipw_pooled` do not show it; their recorded median errors are
about 497 and 749 dollars.

(rerun-with-the-current-build)=
## October 16 AIPW rerun

The default configuration changed after the recorded run. Commit `c011116`, made at
11:24 on October 11, stopped `aipw` from choosing the two-part outcome model automatically
and bounded that model. The run's records show the two-part model for every default AIPW
draw. The default used in the October 16 rerun picks the pooled model. The script
`rerun_aipw_current.py` reruns the six AIPW arms on the same draws, folds and seeds with
the October 16 build, and writes `rerun_aipw_current.json`. The table below reads that file.

In [ ]:
rerun = json.loads(Path("causal_estimators/rerun_aipw_current.json").read_text())
rows = pl.DataFrame(rerun["rows"])
current = (
    rows.with_columns((pl.col("ate_hat_current") - pl.col("ate_truth")).abs().alias("current_abs"))
    .with_columns((pl.col("ate_hat_recorded") - pl.col("ate_truth")).abs().alias("recorded_abs"))
    .group_by("arm")
    .agg(
        pl.col("recorded_abs").median().alias("recorded_median"),
        pl.col("current_abs").median().alias("current_median"),
        pl.col("recorded_abs").max().alias("recorded_largest"),
        pl.col("current_abs").max().alias("current_largest"),
        pl.col("outcome_model").unique().alias("current_outcome_models"),
    )
    .sort("arm")
)
(
    GT(current)
    .tab_header(
        title="AIPW arms on the ten effect draws: recorded run against the current build",
        subtitle="Absolute ATE error in dollars; lower is better; same draws, folds and seeds in both runs",
    )
    .fmt_number(columns=["recorded_median", "current_median", "recorded_largest", "current_largest"], decimals=0)
    .cols_label(
        recorded_median="Recorded median",
        current_median="Current median",
        recorded_largest="Recorded largest",
        current_largest="Current largest",
        current_outcome_models="Current outcome model",
    )
    .tab_stubhead(label="Arm")
    .tab_source_note(
        "Recorded: per_draw_results.jsonl, October 11 build. Current: rerun_aipw_current.json, build at the current "
        "commit. The pooled arm has the same error in both runs, which is a check that the harness is unchanged."
    )
)

Four of the six arms no longer show the recorded failure. The default arm's median error
falls from about 294,000 dollars to about 790 dollars. The no-clipping arm has the largest
remaining error, about 59,700 dollars on one draw. The rerun's default outcome model is
the pooled model in every draw, so the default arm and the pooled arm now differ only in
the propensity handling.

## Contrasts between arms

The protocol fixes sixteen contrasts. Each is the first arm minus the second, on the
ATE bias, the ATE root mean squared error, the 90% coverage, the median
out-of-sample error for individual effects, and the median timing. The table keeps the
ATE RMSE and the timings, and the contrast labels say which two arms are compared.

In [ ]:
results = json.loads(RESULTS.read_text())
contrast_rows = [
    {
        "Group": c["group"],
        "Contrast": c["label"],
        "ATE bias ($)": c["ate_bias"],
        "ATE RMSE ($)": c["ate_rmse"],
        "Median seconds (difference)": c["wall_s_median"],
    }
    for c in results["contrasts"]
]
contrast_table = pl.DataFrame(contrast_rows).sort(["Group", "Contrast"])
(
    GT(contrast_table)
    .tab_header(
        title="Pre-specified contrasts between arms",
        subtitle="Each row is the first arm minus the second arm; ATE bias and RMSE in dollars, fit time in seconds",
    )
    .fmt_number(columns=["ATE bias ($)", "ATE RMSE ($)"], decimals=0)
    .fmt_number(columns=["Median seconds (difference)"], decimals=2)
    .tab_stubhead(label="Contrast")
    .tab_source_note("Groups: a = nuisance learners, b = framework, c = second stage, d = defaults. Source: results.json.")
)

The contrasts that involve the default AIPW arm in the recorded run are large, because
that arm's errors were large in that run. The contrasts among the linear arms and the smooth AIPW arm are in the hundreds to
thousands of dollars. The study applies no significance test to the contrasts. With ten effect draws and no reported uncertainty for these contrasts, the
differences are descriptive magnitudes.

## Findings

- **The historical default AIPW arm failed in the October 11 run and completes in the
  October 16 rerun.** Across the ten effect draws, `mb_aipw_ate` estimates ranged from about 16,500
  to 19.1 million dollars in the October 11 build, against a truth near 4,270. In the October 16 rerun, the same `smooth=False` arm has a median error of about
  790 dollars and a largest error of about 2,320. The default changed between the two
  builds, so the recorded failure is a defect of the earlier build. The no-clipping
  variant still has one draw with a large error, about 59,700 dollars.
- **mboostrs' smooth AIPW is competitive with the EconML and hand-built linear arms.**
  Its median error is about 497 dollars, against 331 for EconML's linear arm with HGB
  nuisances.
- **The nuisance learners account for part of the gap among the linear arms.** GAM
  nuisances and HGB nuisances give median errors within about 140 dollars of each
  other, and the GAM arms are much faster.
- **The hand-built R-learner matches EconML's linear arm exactly on HGB nuisances.**
  Their error figures are identical, which is what the shared design predicts.
- **Nuisance choice changes EconML's fit time by about a factor of 28.** Its linear arm
  takes a median of 0.29 seconds with GAM nuisances and 7.97 seconds with HGB nuisances.
  Library speed is therefore confounded with nuisance choice, and this study does not
  separate the two.
- **The coverage figures for the recorded default AIPW arm do not indicate calibration.**
  They come from very wide intervals, as described above.

## Limits

- **Ten effect draws and two null draws.** The medians and extremes are descriptive. No
  significance test is applied.
- **Shared machine.** The load average was 5.5 to 9 at the start of the study, so the
  timings are upper bounds and vary with load. They compare arms within this run only.
- **Different partitions for mboostrs and the other arms.** The folds for mboostrs differ
  from those of the other libraries. The protocol records this.
- **DoubleML's response weights.** Its score uses the weights, but its nuisance fits do
  not. The protocol records this too.
- **Per-row intervals describe second-stage variance only.** They do not cover the bias
  of the nuisance fits, so their coverage is expected to fall below 90%.
- **Two builds.** The recorded per-draw results come from the library build of October 11.
  The rerun uses the October 16 build. The two builds differ in the AIPW
  defaults, so the arms in the two tables were not run on the same code. The pooled arm
  gives the same median error in both, 749 dollars, because its path did not change.
- **The protocol has an erratum.** The response-weight range was written before the
  weights were inspected. The observed range is 1.045 to 1.654. No estimator, setting,
  fold, seed or draw changed.

## Reproduction

From the repository root, with the environment in the protocol's requirements:

```bash
python docs/book/benchmarks/causal_estimators/run_arms.py --src /path/to/CPS1988.csv
python docs/book/benchmarks/causal_estimators/summarize.py
```

A full run takes about 40 minutes on an Apple Silicon laptop.